The packages used in RDC is:
- pandas
- pathlib

In [16]:
import pandas as pd
from pathlib import Path

Dependent Variable - %Change in Manufacturing Employment

In [1]:
import pandas as pd
from pathlib import Path

# Build portable paths relative to this notebook's location
notebook_dir = Path(__file__).parent if "__file__" in dir() else Path().resolve()
project_root = notebook_dir.parent
raw_path = project_root / "data" / "raw" / "Dependent Var" / "Manufacturing Employment %.csv"
clean_dir = project_root / "data" / "clean"
clean_dir.mkdir(parents=True, exist_ok=True)

# Load raw data (skip the 3 header rows, use row 4 as header)
df_raw = pd.read_csv(raw_path, skiprows=3, header=0)
df_raw.rename(columns={df_raw.columns[0]: "Country"}, inplace=True)

# Filter to target countries and years
countries = ["Sweden", "Germany", "United States"]
years = [str(y) for y in range(2019, 2025)]

df_filtered = df_raw[df_raw["Country"].isin(countries)][["Country"] + years].copy()
df_filtered.reset_index(drop=True, inplace=True)
df_filtered[years] = df_filtered[years].apply(pd.to_numeric, errors="coerce")

# --- Table 1: Nominal employment (thousand persons) ---
df_nominal = df_filtered.copy()
df_nominal.columns = ["Country"] + [f"Employment {y} (000s)" for y in years]
df_nominal.to_csv(clean_dir / "Manufacturing Employment Nominal.csv", index=False)

# --- Table 2: Year-on-year % change ---
change_cols = {"Country": df_filtered["Country"].values}
for i in range(1, len(years)):
    prev, curr = years[i - 1], years[i]
    change_cols[f"% Change {prev}-{curr}"] = (
        (df_filtered[curr] - df_filtered[prev]) / df_filtered[prev] * 100
    ).values
df_pct = pd.DataFrame(change_cols)
df_pct.to_csv(clean_dir / "%Change in Manufacturing Employment.csv", index=False)

# --- Display both tables ---
print("=== Table 1: Nominal Employment (000s) ===")
print(df_nominal.to_string(index=False))
print()
print("=== Table 2: % Change in Manufacturing Employment ===")
print(df_pct.to_string(index=False))


=== Table 1: Nominal Employment (000s) ===
      Country  Employment 2019 (000s)  Employment 2020 (000s)  Employment 2021 (000s)  Employment 2022 (000s)  Employment 2023 (000s)  Employment 2024 (000s)
United States                15741.00               14550.000               14718.000                15231.00               15570.000               15023.000
      Germany                 8012.65                7490.000                8097.125                 7966.35                7757.875                7767.000
       Sweden                  513.50                 500.525                 494.925                  492.10                 522.675                 503.825

=== Table 2: % Change in Manufacturing Employment ===
      Country  % Change 2019-2020  % Change 2020-2021  % Change 2021-2022  % Change 2022-2023  % Change 2023-2024
United States           -7.566228            1.154639            3.485528            2.225724           -3.513166
      Germany           -6.522811         

Independent Variable - Union Density  

In [11]:
import pandas as pd
from pathlib import Path

# Build portable paths relative to this notebook's location
notebook_dir = Path(__file__).parent if "__file__" in dir() else Path().resolve()
project_root = notebook_dir.parent
raw_path = project_root / "data" / "raw" / "Independent Var" / "COORD.xlsx"
clean_path = project_root / "data" / "clean" / "Union Density.csv"

# Load Union Density data from ICTWSS workbook (Section F)
df_raw = pd.read_excel(raw_path, sheet_name="Section F.", skiprows=4)

# Filter to target countries and years
countries = ["Sweden", "Germany", "United States"]
years = list(range(2019, 2025))

df_filtered = df_raw[
    df_raw["Country"].isin(countries) & df_raw["Year"].isin(years)
][["Country", "Year", "UD_hist"]].copy()

# Clean values and reshape into a wide table
df_filtered.rename(columns={"UD_hist": "Union Density"}, inplace=True)
df_filtered["Union Density"] = pd.to_numeric(df_filtered["Union Density"], errors="coerce")

df_clean = (
    df_filtered.pivot(index="Country", columns="Year", values="Union Density")
    .reindex(countries)
    .reindex(columns=years)
    .reset_index()
)

df_clean.columns = ["Country"] + [f"Union Density {y}" for y in years]

# Save cleaned table to csv
clean_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(clean_path, index=False)

print("Cleaned table saved to:", clean_path)
print(df_clean.to_string(index=False))


Cleaned table saved to: C:\Users\renee\OneDrive\Desktop\ECC3479\Project\ECC3479_UNITPROJECT\data\clean\Union Density.csv
      Country  Union Density 2019  Union Density 2020  Union Density 2021  Union Density 2022  Union Density 2023  Union Density 2024
       Sweden           65.599998                67.0                67.0                65.5           65.199997           65.900002
      Germany           15.100000                15.1                14.8                14.4           14.300000           14.100000
United States           10.300000                10.8                10.3                10.1           10.000000            9.900000


Independent Variable - Labour Market Coordination (COORD)

In [12]:
import pandas as pd
from pathlib import Path

# Build portable paths relative to this notebook's location
notebook_dir = Path(__file__).parent if "__file__" in dir() else Path().resolve()
project_root = notebook_dir.parent
raw_path = project_root / "data" / "raw" / "Independent Var" / "COORD.xlsx"
clean_path = project_root / "data" / "clean" / "COORD_2019.csv"

# Load Labour Market Coordination data from ICTWSS workbook (Section B)
df_raw = pd.read_excel(raw_path, sheet_name="Section B.", skiprows=4)

# Filter to target countries and year
countries = ["Sweden", "Germany", "United States"]
target_year = 2019

df_clean = df_raw[
    df_raw["Country"].isin(countries) & (df_raw["Year"] == target_year)
][["Country", "Year", "Coord"]].copy()

# Clean values and keep country order
df_clean.rename(columns={"Coord": "Labour Market Coordination (COORD)"}, inplace=True)
df_clean["Labour Market Coordination (COORD)"] = pd.to_numeric(
    df_clean["Labour Market Coordination (COORD)"], errors="coerce"
)
df_clean["Country"] = pd.Categorical(df_clean["Country"], categories=countries, ordered=True)
df_clean = df_clean.sort_values("Country").reset_index(drop=True)

# Save cleaned table to csv
clean_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(clean_path, index=False)

print("Cleaned table saved to:", clean_path)
print(df_clean.to_string(index=False))


Cleaned table saved to: C:\Users\renee\OneDrive\Desktop\ECC3479\Project\ECC3479_UNITPROJECT\data\clean\COORD_2019.csv
      Country  Year  Labour Market Coordination (COORD)
       Sweden  2019                                   4
      Germany  2019                                   4
United States  2019                                   1


Independent Variable - Collective Bargaining Coverage (AdjCov_hist)

In [13]:
import pandas as pd
from pathlib import Path

# Build portable paths relative to this notebook's location
notebook_dir = Path(__file__).parent if "__file__" in dir() else Path().resolve()
project_root = notebook_dir.parent
raw_path = project_root / "data" / "raw" / "Independent Var" / "COORD.xlsx"
clean_path = project_root / "data" / "clean" / "AdjCov_hist.csv"

# Load collective bargaining coverage data from ICTWSS workbook (Section G)
df_raw = pd.read_excel(raw_path, sheet_name="Section G.", skiprows=4)

# Filter to target countries and years
countries = ["Sweden", "Germany", "United States"]
years = list(range(2019, 2025))

df_filtered = df_raw[
    df_raw["Country"].isin(countries) & df_raw["Year"].isin(years)
][["Country", "Year", "AdjCov_hist"]].copy()

# Clean and reshape into a wide table
df_filtered["AdjCov_hist"] = pd.to_numeric(df_filtered["AdjCov_hist"], errors="coerce")
df_clean = (
    df_filtered.pivot(index="Country", columns="Year", values="AdjCov_hist")
    .reindex(countries)
    .reindex(columns=years)
    .reset_index()
)

df_clean.columns = ["Country"] + [f"AdjCov_hist {y}" for y in years]

# Save cleaned table to csv
clean_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(clean_path, index=False)

print("Cleaned table saved to:", clean_path)
print(df_clean.to_string(index=False))


Cleaned table saved to: C:\Users\renee\OneDrive\Desktop\ECC3479\Project\ECC3479_UNITPROJECT\data\clean\AdjCov_hist.csv
      Country  AdjCov_hist 2019  AdjCov_hist 2020  AdjCov_hist 2021  AdjCov_hist 2022  AdjCov_hist 2023  AdjCov_hist 2024
       Sweden         88.699997              89.0              87.0              87.5         87.900002              88.0
      Germany         52.000000              51.0              52.0              51.0         49.000000              49.0
United States         11.600000              12.1              11.6              11.3         11.200000              11.1


Control Variable - Export %(dependency)

In [15]:
import pandas as pd
from pathlib import Path

# Build portable paths relative to this notebook's location
notebook_dir = Path(__file__).parent if "__file__" in dir() else Path().resolve()
project_root = notebook_dir.parent
raw_path = project_root / "data" / "raw" / "Control Var" / "Export % (dependency).xls"
clean_path = project_root / "data" / "clean" / "Export_dependency.csv"

# Load raw data (first 3 rows are metadata, row index 3 is the header)
df_raw = pd.read_excel(raw_path, sheet_name="Data", skiprows=3, header=0)
df_raw.rename(columns={"Country Name": "Country"}, inplace=True)

# Convert column names to strings for consistent year selection
df_raw.columns = [str(c).replace(".0", "") for c in df_raw.columns]

# Filter to target countries and years
countries = ["Sweden", "Germany", "United States"]
years = [str(y) for y in range(2019, 2025)]

df_clean = df_raw[df_raw["Country"].isin(countries)][["Country"] + years].copy()
df_clean[years] = df_clean[years].apply(pd.to_numeric, errors="coerce")

# Keep requested country order
df_clean["Country"] = pd.Categorical(df_clean["Country"], categories=countries, ordered=True)
df_clean = df_clean.sort_values("Country").reset_index(drop=True)

# Rename year columns for clarity
df_clean.columns = ["Country"] + [f"Export Dependency % GDP {y}" for y in years]

# Save cleaned table to csv
clean_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(clean_path, index=False)

print("Cleaned table saved to:", clean_path)
print(df_clean.to_string(index=False))


Cleaned table saved to: C:\Users\renee\OneDrive\Desktop\ECC3479\Project\ECC3479_UNITPROJECT\data\clean\Export_dependency.csv
      Country  Export Dependency % GDP 2019  Export Dependency % GDP 2020  Export Dependency % GDP 2021  Export Dependency % GDP 2022  Export Dependency % GDP 2023  Export Dependency % GDP 2024
       Sweden                     48.288658                     44.287377                     47.484237                     54.312063                     55.195878                     54.309581
      Germany                     42.340244                     39.119807                     42.519838                     45.628404                     42.967262                     41.434059
United States                     11.876871                     10.214106                     10.960417                     11.784405                     11.184413                     11.107457


Control Variable - GDP

In [16]:
import pandas as pd
from pathlib import Path

# Build portable paths relative to this notebook's location
notebook_dir = Path(__file__).parent if "__file__" in dir() else Path().resolve()
project_root = notebook_dir.parent
raw_path = project_root / "data" / "raw" / "Control Var" / "GDP.xls"
clean_path = project_root / "data" / "clean" / "GDP.csv"

# Load raw data (first 3 rows are metadata, row index 3 is the header)
df_raw = pd.read_excel(raw_path, sheet_name="Data", skiprows=3, header=0)
df_raw.rename(columns={"Country Name": "Country"}, inplace=True)

# Convert column names to strings for consistent year selection
df_raw.columns = [str(c).replace(".0", "") for c in df_raw.columns]

# Filter to target countries and years
countries = ["Sweden", "Germany", "United States"]
years = [str(y) for y in range(2019, 2025)]

df_clean = df_raw[df_raw["Country"].isin(countries)][["Country"] + years].copy()
df_clean[years] = df_clean[years].apply(pd.to_numeric, errors="coerce")

# Keep requested country order
df_clean["Country"] = pd.Categorical(df_clean["Country"], categories=countries, ordered=True)
df_clean = df_clean.sort_values("Country").reset_index(drop=True)

# Rename year columns for clarity
df_clean.columns = ["Country"] + [f"GDP Current US$ {y}" for y in years]

# Save cleaned table to csv
clean_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(clean_path, index=False)

print("Cleaned table saved to:", clean_path)
print(df_clean.to_string(index=False))


Cleaned table saved to: C:\Users\renee\OneDrive\Desktop\ECC3479\Project\ECC3479_UNITPROJECT\data\clean\GDP.csv
      Country  GDP Current US$ 2019  GDP Current US$ 2020  GDP Current US$ 2021  GDP Current US$ 2022  GDP Current US$ 2023  GDP Current US$ 2024
       Sweden          5.308941e+11          5.442657e+11          6.316933e+11          5.750712e+11          5.789909e+11          6.037152e+11
      Germany          3.959895e+12          3.941399e+12          4.355252e+12          4.201022e+12          4.562208e+12          4.685593e+12
United States          2.138098e+13          2.106047e+13          2.331508e+13          2.560485e+13          2.729217e+13          2.875096e+13


Control Variable - GDP growth annual

In [17]:
import pandas as pd
from pathlib import Path

# Build portable paths relative to this notebook's location
notebook_dir = Path(__file__).parent if "__file__" in dir() else Path().resolve()
project_root = notebook_dir.parent
raw_path = project_root / "data" / "raw" / "Control Var" / "GDP growth Annual.xls"
clean_path = project_root / "data" / "clean" / "GDP_growth_annual.csv"

# Load raw data (first 3 rows are metadata, row index 3 is the header)
df_raw = pd.read_excel(raw_path, sheet_name="Data", skiprows=3, header=0)
df_raw.rename(columns={"Country Name": "Country"}, inplace=True)

# Convert column names to strings for consistent year selection
df_raw.columns = [str(c).replace(".0", "") for c in df_raw.columns]

# Filter to target countries and years
countries = ["Sweden", "Germany", "United States"]
years = [str(y) for y in range(2019, 2025)]

df_clean = df_raw[df_raw["Country"].isin(countries)][["Country"] + years].copy()
df_clean[years] = df_clean[years].apply(pd.to_numeric, errors="coerce")

# Keep requested country order
df_clean["Country"] = pd.Categorical(df_clean["Country"], categories=countries, ordered=True)
df_clean = df_clean.sort_values("Country").reset_index(drop=True)

# Rename year columns for clarity
df_clean.columns = ["Country"] + [f"GDP Growth Annual % {y}" for y in years]

# Save cleaned table to csv
clean_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(clean_path, index=False)

print("Cleaned table saved to:", clean_path)
print(df_clean.to_string(index=False))


Cleaned table saved to: C:\Users\renee\OneDrive\Desktop\ECC3479\Project\ECC3479_UNITPROJECT\data\clean\GDP_growth_annual.csv
      Country  GDP Growth Annual % 2019  GDP Growth Annual % 2020  GDP Growth Annual % 2021  GDP Growth Annual % 2022  GDP Growth Annual % 2023  GDP Growth Annual % 2024
       Sweden                  2.608230                 -1.933797                  5.225864                  1.255436                 -0.204061                  0.820072
      Germany                  0.977735                 -4.131914                  3.910000                  1.809258                 -0.869648                 -0.495852
United States                  2.583825                 -2.163029                  6.055053                  2.512375                  2.887556                  2.793001


Control Variable - %Manufacturing Value Added (from Dependent Var source)

In [ ]:
import pandas as pd
from pathlib import Path

# Build portable paths relative to this notebook's location
notebook_dir = Path(__file__).parent if "__file__" in dir() else Path().resolve()
project_root = notebook_dir.parent
raw_path = project_root / "data" / "raw" / "Control Var" / "Manufacturing Value Added.xls"
clean_dir = project_root / "data" / "clean"
clean_dir.mkdir(parents=True, exist_ok=True)

# Load raw data (first 3 rows are metadata, row index 3 is the header)
df_raw = pd.read_excel(raw_path, sheet_name="Data", skiprows=3, header=0)
df_raw.rename(columns={"Country Name": "Country"}, inplace=True)

# Filter to target countries and years
countries = ["Sweden", "Germany", "United States"]
years = [str(y) for y in range(2019, 2025)]

df_filtered = df_raw[df_raw["Country"].isin(countries)][["Country"] + years].copy()
df_filtered.reset_index(drop=True, inplace=True)
df_filtered[years] = df_filtered[years].apply(pd.to_numeric, errors="coerce")

# --- Table 1: Nominal values (Manufacturing Value Added % of GDP) ---
df_nominal = df_filtered.copy()
df_nominal.columns = ["Country"] + [f"Mfg Value Added % GDP {y}" for y in years]
df_nominal.to_csv(clean_dir / "%Manufacturing Value Added Nominal.csv", index=False)

# --- Table 2: Year-on-year % change ---
change_cols = {"Country": df_filtered["Country"].values}
for i in range(1, len(years)):
    prev, curr = years[i - 1], years[i]
    change_cols[f"% Change {prev}-{curr}"] = (
        (df_filtered[curr] - df_filtered[prev]) / df_filtered[prev] * 100
    ).values
df_pct = pd.DataFrame(change_cols)
df_pct.to_csv(clean_dir / "%Manufacturing Value Added Change.csv", index=False)

# --- Display both tables ---
print("=== Table 1: Manufacturing Value Added (% of GDP) ===")
print(df_nominal.to_string(index=False))
print()
print("=== Table 2: % Change in Manufacturing Value Added ===")
print(df_pct.to_string(index=False))

=== Manufacturing Value Added (VAPGDPMA 1) ===
Note: VAPGDPMA (1) source is a US-only series; Sweden and Germany remain missing.
      Country VAPGDPMA(1) 2019 VAPGDPMA(1) 2020 VAPGDPMA(1) 2021 VAPGDPMA(1) 2022 VAPGDPMA(1) 2023 VAPGDPMA(1) 2024
       Sweden             <NA>             <NA>             <NA>             <NA>             <NA>             <NA>
      Germany             <NA>             <NA>             <NA>             <NA>             <NA>             <NA>
United States             10.6             10.1             10.2             10.2             10.1              9.8


Control Variable - %Manufacturing Value Added (US) - For Future Robustness Checks

In [15]:
import pandas as pd
from pathlib import Path

# Build portable paths relative to this notebook's location
notebook_dir = Path(__file__).parent if "__file__" in dir() else Path().resolve()
project_root = notebook_dir.parent
raw_path = project_root / "data" / "raw" / "Control Var" / "VAPGDPMA (1).xlsx"
clean_path = project_root / "data" / "clean" / "VAPGDPMA_1.csv"

countries = ["Sweden", "Germany", "United States"]
years = list(range(2019, 2025))

# FRED file structure: country information is not included; this series is US only
df_raw = pd.read_excel(raw_path, sheet_name="Annual")
df_raw["observation_date"] = pd.to_datetime(df_raw["observation_date"], errors="coerce")
df_raw["Year"] = df_raw["observation_date"].dt.year
df_raw["VAPGDPMA"] = pd.to_numeric(df_raw["VAPGDPMA"], errors="coerce")

# Keep only requested years and reshape to one-row wide US table
us_years = df_raw[df_raw["Year"].isin(years)][["Year", "VAPGDPMA"]].drop_duplicates("Year")
us_wide = us_years.set_index("Year").T
us_wide.index = ["United States"]
us_wide = us_wide.reindex(columns=years)

# Build requested 3-country table (missing values where source data is unavailable)
df_clean = pd.DataFrame({"Country": countries})
for y in years:
    df_clean[f"VAPGDPMA(1) {y}"] = pd.NA

for y in years:
    df_clean.loc[df_clean["Country"] == "United States", f"VAPGDPMA(1) {y}"] = us_wide.loc["United States", y]

# Save cleaned table to csv
clean_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(clean_path, index=False)

print("Cleaned table saved to:", clean_path)
print("Note: VAPGDPMA (1) source is a US-only FRED series; Sweden and Germany remain missing.")
print(df_clean.to_string(index=False))

Cleaned table saved to: C:\Users\renee\OneDrive\Desktop\ECC3479\Project\ECC3479_UNITPROJECT\data\clean\VAPGDPMA_1.csv
Note: VAPGDPMA (1) source is a US-only FRED series; Sweden and Germany remain missing.
      Country VAPGDPMA(1) 2019 VAPGDPMA(1) 2020 VAPGDPMA(1) 2021 VAPGDPMA(1) 2022 VAPGDPMA(1) 2023 VAPGDPMA(1) 2024
       Sweden             <NA>             <NA>             <NA>             <NA>             <NA>             <NA>
      Germany             <NA>             <NA>             <NA>             <NA>             <NA>             <NA>
United States             10.6             10.1             10.2             10.2             10.1              9.8
